# BERT Signal Generation: FinBERT · MiniLM · BERTopic

Generates three transformer-based NLP signals for comparison against the LM-dictionary
and TF-IDF baselines from `nlp_features.ipynb`.

| Track | Model | Signal |
|---|---|---|
| A | `yiyanghkust/finbert-tone` | Net sentiment per (cik, year); YoY tone change |
| B | `all-MiniLM-L6-v2` | YoY cosine similarity of 384-dim embeddings |
| C | BERTopic | YoY shift in topic distribution |

**Requires** GPU runtime (`Runtime → Change runtime type → T4 GPU`).
All tracks checkpoint to `/content/temp_bert/` and resume automatically.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 1. Configuration

In [ ]:
import os

CONFIG = {
    'drive_folder':       '/content/drive/MyDrive/FML_project_4',
    'chunks_path':        '/content/drive/MyDrive/FML_project_4/text_chunks.parquet',
    'preprocessed_path':  '/content/drive/MyDrive/FML_project_4/text_preprocessed.parquet',
    'usable_pairs_path':  '/content/drive/MyDrive/FML_project_4/usable_pairs.parquet',
    'analysis_panel_path':'/content/drive/MyDrive/FML_project_4/analysis_panel.parquet',
    'output_folder':      '/content/drive/MyDrive/FML_project_4',

    # Models
    # finbert-tone (Huang et al. 2023, CAR): fine-tuned on 10-K filings + analyst text.
    # Do NOT use ProsusAI/finbert — trained on Financial PhraseBank (short analyst headlines,
    # not MD&A prose), so label probabilities are poorly calibrated for 10-K language.
    'finbert_model':  'yiyanghkust/finbert-tone',
    'minilm_model':   'sentence-transformers/all-MiniLM-L6-v2',

    # Sections to use for all BERT tracks
    'sections': ['item_1', 'item_7'],

    # Inference batch sizes — reduce if GPU OOM
    'finbert_batch': 32,
    'minilm_batch':  64,

    # CIKs per checkpoint file
    'cik_batch': 50,
}
print('Config OK')

## 2. Install dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers', 'sentence-transformers', 'bertopic',
    'pyarrow', 'torch', 'tqdm', 'umap-learn', 'hdbscan'], check=True)
print('Dependencies ready')

## 3. Load data

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
import glob as _glob
import pyarrow.parquet as pq
import pyarrow as pa
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cpu':
    print('WARNING: No GPU detected. Embedding tracks will be very slow.')

# Load usable pairs filter
usable = pd.read_parquet(CONFIG['usable_pairs_path'])
usable['cik']  = usable['cik'].astype(str).str.lstrip('0')
usable['year'] = usable['year'].astype('Int64')
usable_set     = set(zip(usable['cik'], usable['year'].astype(int)))
print(f'Usable (cik, year) pairs: {len(usable_set):,}')

# Load chunks — item_1 and item_7 only
print('Loading text_chunks.parquet ...')
chunks = pd.read_parquet(CONFIG['chunks_path'])
chunks['cik']  = chunks['cik'].astype(str).str.lstrip('0')
chunks['year'] = chunks['year'].astype('Int64')
chunks = chunks[chunks['section'].isin(CONFIG['sections'])]
chunks = chunks[chunks.apply(lambda r: (r['cik'], int(r['year'])) in usable_set, axis=1)]
print(f'Chunks after filter: {len(chunks):,}  ({chunks["cik"].nunique():,} CIKs)')

ALL_CIKS = sorted(chunks['cik'].unique().tolist())

## 4. Track A — FinBERT-Tone Sentiment

Each chunk → P(positive), P(negative), P(neutral) using `yiyanghkust/finbert-tone`
(Huang et al. 2023, *Contemporary Accounting Research*). Fine-tuned on 10-K filings
and analyst reports — more appropriate for MD&A language than ProsusAI/finbert,
which was trained only on short analyst headlines.

Aggregated to (cik, year) by averaging across all chunks. Signal: `finbert_net = mean_pos − mean_neg`.

**Resume**: checkpoints every 50 CIKs to `/content/temp_bert/finbert/`.

In [ ]:
FINBERT_PATH = os.path.join(CONFIG['output_folder'], 'finbert_sentiment.parquet')
TEMP_FB      = '/content/temp_bert/finbert'
os.makedirs(TEMP_FB, exist_ok=True)

if os.path.exists(FINBERT_PATH):
    print('finbert_sentiment.parquet exists — skipping Track A.')
    finbert_df = pd.read_parquet(FINBERT_PATH)
else:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    fb_tok = AutoTokenizer.from_pretrained(CONFIG['finbert_model'])
    fb_mod = AutoModelForSequenceClassification.from_pretrained(CONFIG['finbert_model'])
    fb_mod.eval().to(DEVICE)

    # Read label order from model config — avoids hardcoding positional assumptions
    # finbert-tone labels are typically: positive, negative, neutral (but may vary)
    id2label: dict[int, str] = {int(k): v.lower() for k, v in fb_mod.config.id2label.items()}
    pos_idx = next(i for i, lbl in id2label.items() if 'pos' in lbl)
    neg_idx = next(i for i, lbl in id2label.items() if 'neg' in lbl)
    neu_idx = next(i for i, lbl in id2label.items() if 'neu' in lbl)
    print(f'Label order: {id2label}  pos={pos_idx} neg={neg_idx} neu={neu_idx}')

    def score_chunks(texts: list[str]) -> np.ndarray:
        """Returns (n, 3) array: [pos_prob, neg_prob, neu_prob] per text."""
        enc = fb_tok(texts, padding=True, truncation=True,
                     max_length=512, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            logits = fb_mod(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        # Reorder to [pos, neg, neu] regardless of model's internal label order
        return np.stack([probs[:, pos_idx], probs[:, neg_idx], probs[:, neu_idx]], axis=1)

    cik_batches = [ALL_CIKS[i:i+CONFIG['cik_batch']]
                   for i in range(0, len(ALL_CIKS), CONFIG['cik_batch'])]
    done = {int(os.path.basename(f)[8:12])
            for f in _glob.glob(f'{TEMP_FB}/finbert_*.parquet')}
    print(f'{len(cik_batches)} batches — {len(done)} done — {len(cik_batches)-len(done)} remaining')

    for b_num, batch_ciks in enumerate(tqdm(cik_batches, desc='FinBERT')):
        tmp = f'{TEMP_FB}/finbert_{b_num:04d}.parquet'
        if os.path.exists(tmp):
            continue

        sub    = chunks[chunks['cik'].isin(batch_ciks)]
        texts  = sub['text'].tolist()
        B      = CONFIG['finbert_batch']
        probs  = np.vstack([score_chunks(texts[i:i+B]) for i in range(0, len(texts), B)])

        result = sub[['cik', 'year']].copy().reset_index(drop=True)
        result['fb_pos']     = probs[:, 0]
        result['fb_neg']     = probs[:, 1]
        result['fb_neutral'] = probs[:, 2]

        # Aggregate to (cik, year): mean probabilities across all chunks
        agg = result.groupby(['cik', 'year'])[['fb_pos','fb_neg','fb_neutral']].mean().reset_index()
        agg['finbert_net'] = agg['fb_pos'] - agg['fb_neg']
        agg.to_parquet(tmp, index=False, compression='snappy')
        del sub, texts, probs, result, agg; gc.collect()

    # Merge + YoY change
    files = sorted(f for f in _glob.glob(f'{TEMP_FB}/finbert_*.parquet')
                   if os.path.getsize(f) > 0)
    raw = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    raw = raw.sort_values(['cik', 'year'])
    raw['finbert_tone_change'] = raw.groupby('cik')['finbert_net'].diff()
    raw.to_parquet(FINBERT_PATH, index=False, compression='snappy')
    finbert_df = raw
    print(f'Saved finbert_sentiment.parquet  {len(finbert_df):,} rows')

finbert_df.head(3)

## 5. Track B — MiniLM Embeddings → YoY Cosine Similarity

Each chunk → 384-dim embedding. Averaged per (cik, year) → firm-year vector.
YoY cosine distance = `1 − cos(emb_t, emb_{t−1})`.

**Resume**: checkpoints every 50 CIKs.

In [ ]:
MINILM_PATH = os.path.join(CONFIG['output_folder'], 'minilm_similarity.parquet')
TEMP_ML     = '/content/temp_bert/minilm'
os.makedirs(TEMP_ML, exist_ok=True)

if os.path.exists(MINILM_PATH):
    print('minilm_similarity.parquet exists — skipping Track B.')
    minilm_df = pd.read_parquet(MINILM_PATH)
else:
    from sentence_transformers import SentenceTransformer

    ml_model = SentenceTransformer(CONFIG['minilm_model'], device=DEVICE)

    cik_batches = [ALL_CIKS[i:i+CONFIG['cik_batch']]
                   for i in range(0, len(ALL_CIKS), CONFIG['cik_batch'])]
    done = {int(os.path.basename(f)[7:11])
            for f in _glob.glob(f'{TEMP_ML}/minilm_*.parquet')}
    print(f'{len(cik_batches)} batches — {len(done)} done — {len(cik_batches)-len(done)} remaining')

    for b_num, batch_ciks in enumerate(tqdm(cik_batches, desc='MiniLM')):
        tmp = f'{TEMP_ML}/minilm_{b_num:04d}.parquet'
        if os.path.exists(tmp):
            continue

        sub   = chunks[chunks['cik'].isin(batch_ciks)]
        embs  = ml_model.encode(sub['text'].tolist(),
                                batch_size=CONFIG['minilm_batch'],
                                show_progress_bar=False,
                                convert_to_numpy=True)  # (n_chunks, 384)

        # Store embedding as list column for parquet
        result = sub[['cik','year']].copy().reset_index(drop=True)
        result['emb'] = list(embs)

        # Aggregate: mean embedding per (cik, year)
        agg_rows: list[dict] = []
        for (cik, year), grp in result.groupby(['cik','year']):
            agg_rows.append({'cik': cik, 'year': year,
                             'emb': np.stack(grp['emb'].values).mean(axis=0)})
        agg = pd.DataFrame(agg_rows)
        agg['emb'] = agg['emb'].apply(lambda x: x.tolist())   # JSON-serialisable
        agg.to_parquet(tmp, index=False, compression='snappy')
        del sub, embs, result, agg, agg_rows; gc.collect()

    # Merge and compute YoY cosine similarity
    files = sorted(f for f in _glob.glob(f'{TEMP_ML}/minilm_*.parquet')
                   if os.path.getsize(f) > 0)
    raw = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    raw = raw.sort_values(['cik','year'])

    sim_rows: list[dict] = []
    for cik, grp in raw.groupby('cik'):
        grp = grp.sort_values('year').reset_index(drop=True)
        emb_mat = np.vstack(grp['emb'].tolist())
        for i in range(1, len(grp)):
            sim = float(cosine_similarity([emb_mat[i]], [emb_mat[i-1]])[0, 0])
            sim_rows.append({'cik': cik, 'year': int(grp['year'].iloc[i]),
                             'minilm_sim': sim, 'minilm_change': 1.0 - sim})

    minilm_df = pd.DataFrame(sim_rows)
    minilm_df['year'] = minilm_df['year'].astype('Int64')
    minilm_df.to_parquet(MINILM_PATH, index=False, compression='snappy')
    print(f'Saved minilm_similarity.parquet  {len(minilm_df):,} rows')

minilm_df.head(3)

## 6. Track C — BERTopic: YoY Topic Shift

Fits BERTopic on the full corpus of (item_1 + item_7) clean text. Each (cik, year) gets
a topic probability distribution. YoY cosine distance of those distributions = topic shift.

Uses CPU-friendly UMAP + HDBSCAN. **Resume**: skipped if `bertopic_shift.parquet` exists.

In [ ]:
BERTOPIC_PATH = os.path.join(CONFIG['output_folder'], 'bertopic_shift.parquet')

if os.path.exists(BERTOPIC_PATH):
    print('bertopic_shift.parquet exists — skipping Track C.')
    bertopic_df = pd.read_parquet(BERTOPIC_PATH)
else:
    from bertopic import BERTopic
    from umap import UMAP
    from hdbscan import HDBSCAN

    # Load clean text (not chunks) — one document per (cik, year)
    pre = pd.read_parquet(CONFIG['preprocessed_path'],
                          columns=['cik','year','item_1_clean','item_7_clean'])
    pre['cik']  = pre['cik'].astype(str).str.lstrip('0')
    pre['year'] = pre['year'].astype('Int64')
    pre = pre.merge(usable, on=['cik','year'], how='inner').reset_index(drop=True)

    # Combine sections; cap at 30k chars to control UMAP memory
    pre['doc'] = (pre['item_1_clean'].str[:15_000] + ' ' +
                  pre['item_7_clean'].str[:15_000])

    docs = pre['doc'].tolist()
    print(f'Fitting BERTopic on {len(docs):,} documents ...')

    umap_model  = UMAP(n_components=5, n_neighbors=15, min_dist=0.0,
                       metric='cosine', random_state=42)
    hdbscan_model = HDBSCAN(min_cluster_size=20, metric='euclidean',
                            cluster_selection_method='eom', prediction_data=True)
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        calculate_probabilities=True,
        verbose=True,
    )
    topics, probs = topic_model.fit_transform(docs)
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    print(f'Topics found: {n_topics}')

    # probs shape: (n_docs, n_topics)
    prob_df = pd.DataFrame(probs, columns=[f'topic_{i}' for i in range(probs.shape[1])])
    prob_df = pd.concat([pre[['cik','year']].reset_index(drop=True), prob_df], axis=1)

    # YoY cosine distance between topic distributions
    topic_cols = [c for c in prob_df.columns if c.startswith('topic_')]
    shift_rows: list[dict] = []
    for cik, grp in prob_df.groupby('cik'):
        grp = grp.sort_values('year').reset_index(drop=True)
        mat = grp[topic_cols].values
        for i in range(1, len(grp)):
            sim = float(cosine_similarity([mat[i]], [mat[i-1]])[0, 0])
            shift_rows.append({'cik': cik, 'year': int(grp['year'].iloc[i]),
                               'topic_sim': sim, 'topic_shift': 1.0 - sim})

    bertopic_df = pd.DataFrame(shift_rows)
    bertopic_df['year'] = bertopic_df['year'].astype('Int64')
    bertopic_df.to_parquet(BERTOPIC_PATH, index=False, compression='snappy')
    del pre, docs, prob_df; gc.collect()
    print(f'Saved bertopic_shift.parquet  {len(bertopic_df):,} rows')

bertopic_df.head(3)

## 7. Merge BERT signals → `bert_signals.parquet`

In [ ]:
BERT_SIG_PATH = os.path.join(CONFIG['output_folder'], 'bert_signals.parquet')

bert_signals = (
    finbert_df
    .merge(minilm_df,  on=['cik','year'], how='outer')
    .merge(bertopic_df, on=['cik','year'], how='outer')
)
bert_signals['cik']  = bert_signals['cik'].astype(str).str.lstrip('0')
bert_signals['year'] = bert_signals['year'].astype('Int64')
bert_signals.to_parquet(BERT_SIG_PATH, index=False, compression='snappy')

print(f'bert_signals.parquet: {bert_signals.shape}')
print(f'Columns: {list(bert_signals.columns)}')
bert_signals.head(3)

## 8. Signal sorting diagnostic

Quintile-sort each signal per year, compute mean buy-and-hold return over the
12-month forward window. Long Q5 − Short Q1 reported for each model side-by-side.

**This is a diagnostic only** — it shows monotonic pattern (or lack thereof) in
signal–return correlations. It is NOT a portfolio backtest:
- No transaction costs, rebalancing, or position sizing
- No risk-factor adjustment (market, size, value, momentum)
- Standard errors are not Newey-West corrected

A proper comparison requires Fama-MacBeth regressions with controls.

In [ ]:
import matplotlib.pyplot as plt

panel = pd.read_parquet(CONFIG['analysis_panel_path'])
panel['cik']  = panel['cik'].astype(str).str.lstrip('0')
panel['year'] = panel['year'].astype('Int64')

# Merge BERT signals into panel
panel = panel.merge(bert_signals, on=['cik','year'], how='left')

SIGNALS = {
    'finbert_net':       'FinBERT Sentiment',
    'finbert_tone_change': 'FinBERT Tone Change',
    'minilm_change':     'MiniLM Text Change',
    'topic_shift':       'BERTopic Shift',
}

# Annual returns per (cik, year)
bahr = (
    panel.groupby(['cik','year'])
         .apply(lambda g: (1 + g['monthly_return'].fillna(0)).prod() - 1)
         .reset_index(name='annual_return')
)

results: list[dict] = []
for signal, label in SIGNALS.items():
    if signal not in panel.columns:
        print(f'  Skip {signal} (not in panel)')
        continue
    sig_df = (
        panel.drop_duplicates(['cik','year'])[['cik','year', signal]]
             .merge(bahr, on=['cik','year'])
             .dropna(subset=[signal,'annual_return'])
    )
    sig_df['q'] = sig_df.groupby('year')[signal].transform(
        lambda x: pd.qcut(x, 5, labels=False, duplicates='drop'))
    q_ret = sig_df.groupby('q')['annual_return'].mean()
    ls    = (q_ret.get(4, np.nan) - q_ret.get(0, np.nan)) * 100
    results.append({'Signal': label, 'L/S Return (%)': round(ls, 2),
                    'Q1 (%)': round(q_ret.get(0, np.nan)*100, 2),
                    'Q5 (%)': round(q_ret.get(4, np.nan)*100, 2),
                    'N': len(sig_df)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ca02c' if v >= 0 else '#d62728' for v in results_df['L/S Return (%)']]
ax.bar(results_df['Signal'], results_df['L/S Return (%)'], color=colors, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Long–Short Q5−Q1 Annual Return by Signal', fontsize=13)
ax.set_ylabel('Mean Annual Return (%)')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'bert_signal_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()